<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 7 </h1></center>
<center><h2> Les API en conditions réelles </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 7 · 15 min · < 0,01 $ · prépare le Lab 3**

In [1]:
import datetime
import logging
import math
import os
import time

import httpx
import openai
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()
gateway = {"base_url": os.environ["LLM_BASE_URL"], "api_key": os.environ["LLM_API_KEY"],
           "include_response_headers": True}
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, **gateway)
print("Prêt")

Prêt


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Le streaming </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- La même question, **sans puis avec** streaming.

In [2]:
QUESTION = "Expliquez en cinq phrases comment un salarié acquiert ses congés payés."

start = time.perf_counter()
llm.invoke(QUESTION)
print("invoke : rien pendant", round(time.perf_counter() - start, 2), "s, puis tout d'un coup")

start = time.perf_counter()
chunks = []
for chunk in llm.stream(QUESTION):
    if not chunks:
        print("stream : premier morceau après", round(time.perf_counter() - start, 2), "s")
    chunks.append(chunk.content)
print("stream : fin après", round(time.perf_counter() - start, 2), "s,", len(chunks), "morceaux")
print("les dix premiers :", chunks[:10])

invoke : rien pendant 2.01 s, puis tout d'un coup


stream : premier morceau après 0.45 s


stream : fin après 2.0 s, 172 morceaux
les dix premiers : ['', 'Un', ' salarié', ' acqu', 'iert', ' des', ' cong', 'és', ' pay', 'és']


- Même durée totale, **premier morceau en moins d'une seconde**.
- Les morceaux sont **des tokens**, pas des mots.
- Le streaming **ne rend pas plus rapide** : il rend l'attente visible.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Une vraie erreur 429 </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

![](assets/07_retry.png)

- Un faux réseau devant la gateway : **2 × `429` avec `Retry-After: 1`**, puis il laisse passer.
- Le journal de la bibliothèque OpenAI est **activé**.

In [3]:
def flaky_http_client(failures, status):
    """HTTP client that answers `failures` times with `status`, then forwards to the real gateway."""
    real_transport = httpx.HTTPTransport()
    calls = {"count": 0}

    def handler(request):
        calls["count"] += 1
        if calls["count"] <= failures:
            return httpx.Response(status, headers={"Retry-After": "1"}, json={"error": {"message": f"Erreur {status} simulée"}})
        return real_transport.handle_request(request)

    return httpx.Client(transport=httpx.MockTransport(handler))


logging.basicConfig(format="    [openai] %(message)s")
logging.getLogger("openai").setLevel(logging.INFO)

In [4]:
for max_retries in [0, 3]:
    model = ChatOpenAI(model="gpt-4o-mini", max_retries=max_retries,
                       http_client=flaky_http_client(failures=2, status=429), **gateway)
    start = time.perf_counter()
    try:
        answer = model.invoke("Répondez seulement : reçu.").content
        print(f"max_retries={max_retries} : réponse « {answer} » après {time.perf_counter() - start:.1f} s")
    except openai.RateLimitError as error:
        print(f"max_retries={max_retries} : échec immédiat, {type(error).__name__} {error.status_code}")

max_retries=0 : échec immédiat, OpenAIRateLimitError 429


max_retries=3 : réponse « Reçu. » après 2.6 s


- `max_retries=0` : **échec au premier refus**.
- `max_retries=3` : la bibliothèque **lit `Retry-After`**, attend, réessaie → réponse en ≈ 2,5 s, **sans erreur visible**.
- C'est le **comportement par défaut** de `ChatOpenAI` (2 essais). Au Lab 3, on le coupe pour écrire le nôtre.

- La même chose avec une **erreur 400**.

In [5]:
model = ChatOpenAI(model="gpt-4o-mini", max_retries=3,
                   http_client=flaky_http_client(failures=2, status=400), **gateway)
start = time.perf_counter()
try:
    model.invoke("Bonjour")
except openai.BadRequestError as error:
    print(f"400 : {type(error).__name__}, abandon après {time.perf_counter() - start:.2f} s, sans réessai")
logging.getLogger("openai").setLevel(logging.WARNING)

400 : OpenAIInvalidRequestError, abandon après 0.00 s, sans réessai


- **Aucun réessai** : une requête mal formée le restera.
- On réessaie **429 et 5xx**, jamais 400, 401 ou 404.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. Une boucle d'appel d'outils </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

![](assets/07_tool_loop.png)

- Deux outils, une question qui a besoin **des deux**.

In [6]:
@tool
def accrued_leave(months_worked: int) -> int:
    """Jours ouvrables de congés payés acquis pour un nombre de mois de travail effectif (2,5 par mois, 30 au plus)."""
    return min(30, math.ceil(2.5 * months_worked))


@tool
def working_days(start_date: str, end_date: str) -> int:
    """Nombre de jours ouvrables (du lundi au samedi) entre deux dates incluses, au format AAAA-MM-JJ, hors jours fériés."""
    day = datetime.date.fromisoformat(start_date)
    end = datetime.date.fromisoformat(end_date)
    count = 0
    while day <= end:
        if day.weekday() < 6:
            count += 1
        day += datetime.timedelta(days=1)
    return count


TOOLS = {"accrued_leave": accrued_leave, "working_days": working_days}


def run_with_tools(question, max_turns=5):
    """Calls the model, runs the tools it asks for, and loops until it answers in text."""
    model = llm.bind_tools(list(TOOLS.values()))
    messages = [HumanMessage(question)]
    for turn in range(max_turns):
        response = model.invoke(messages)
        messages.append(response)
        cost = response.response_metadata["headers"]["x-litellm-response-cost"]
        print(f"tour {turn + 1} : {response.usage_metadata['input_tokens']} tokens d'entrée, {cost} $")
        if not response.tool_calls:
            return response.content
        for tool_call in response.tool_calls:
            result = TOOLS[tool_call["name"]].invoke(tool_call)
            print(f"   outil {tool_call['name']}({tool_call['args']}) -> {result.content}")
            messages.append(result)
    return "Nombre maximal de tours atteint."


print(run_with_tools("J'ai travaillé 10 mois. Je pose mes congés du lundi 2 au samedi 14 août 2027, "
                     "sans jour férié sur la période. Combien de jours de congés me restera-t-il ?"))

tour 1 : 167 tokens d'entrée, 6.405e-05 $
   outil accrued_leave({'months_worked': 10}) -> 25
   outil working_days({'start_date': '2027-08-02', 'end_date': '2027-08-14'}) -> 12


tour 2 : 248 tokens d'entrée, 8.34e-05 $
Après avoir travaillé 10 mois, vous avez acquis 25 jours de congés payés. 

En posant vos congés du lundi 2 au samedi 14 août 2027, vous utilisez 12 jours de congés. 

Il vous restera donc :

\[ 25 - 12 = 13 \]

Vous aurez 13 jours de congés restants.


- Les deux outils demandés **dans le même tour**.
- Tokens d'entrée **en hausse à chaque tour** : chaque tour renvoie toute la conversation.
- `max_turns` = **garde-fou** contre une boucle (et une facture) sans fin.
- Cette boucle = **le cœur d'un agent** → `create_agent` au J3.

**→ Lab 3** : un réessai écrit à la main, un appel d'outil pas à pas, puis les deux assemblés.